# Setup Neo4j knowledge graph + SQLite app DB

One-shot bootstrap for a fresh PropertyLens checkout. Reuses the **layer-06 builder** so the graph schema stays in lockstep with `notebooks/06_search_layer/yc_property_search.py`.

## What gets seeded

**Neo4j (graph schema mirrors `notebooks/06_search_layer/01_build_knowledge_base.ipynb`):**

```
(:Property)  ─[:LOCATED_IN]───────────────────►  (:Town)
(:Property)  ─[:NEAREST_FAMOUS_SCHOOL {dist}]──►  (:FamousSchool)
(:Property)  ─[:NEAR_FAMOUS_SCHOOL {dist}]─────►  (:FamousSchool)   ← within 2 km
```

- `Property` (~9,710): every unique `address_key` from the latest feature table, with all 12 normalized 0–10 score dimensions + raw fields (resale_price, floor_area_sqm, dist_*, etc.)
- `Town` (26): `name` per `backend/hdb_towns.py`
- `FamousSchool` (~17): MOE primaries flagged as autonomous / gifted / SAP

**SQLite (`data/propertylens.db` — auth + wishlist + history):**
- All tables created via `Base.metadata.create_all`
- Demo user `user` / `1234` seeded via `ensure_demo_user()`

**Idempotent.** The Neo4j push uses `MERGE`; the demo user creation is a no-op when present.

## Prereqs

- `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `NEO4J_DATABASE` in `.env` (matches `06_search_layer` convention).
- Either a built KB parquet at `notebooks/06_search_layer/artifacts/property_knowledge_base_*.parquet` (fastest path), **or** the source data needed to build it from scratch:
  - `data/feature_data/02_feature_layer/training/outputs/hdb_feature_table_*.csv` (downloaded via `download_artifacts_from_hf.ipynb`)
  - `notebooks/01_data_layer/raw/schools/moe_general_information_of_schools_*.csv`
  - `notebooks/01_data_layer/raw/google_geo/moe_school_geocode_*.csv`
  - `notebooks/01_data_layer/raw/google_geo/hdb_geo_accessibility_noise_features_*.csv`
- Backend deps installed (`pip install -r requirements.txt` for `neo4j`, `sqlalchemy`, `bcrypt`).

## 1 — Setup

In [ ]:
%pip install -q neo4j pandas pyarrow python-dotenv sqlalchemy bcrypt

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve()
load_dotenv(REPO_ROOT / ".env")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Layer-06 builder package — re-used so the graph schema stays canonical.
LAYER_DIR = REPO_ROOT / "notebooks" / "06_search_layer"
if str(LAYER_DIR) not in sys.path:
    sys.path.insert(0, str(LAYER_DIR))

# Layer-06 reads the ‘USERNAME’ env var; chat.py reads ‘USER’. Mirror them so
# both work without forcing the user to set two variables.
if not os.environ.get("NEO4J_USER") and os.environ.get("NEO4J_USERNAME"):
    os.environ["NEO4J_USER"] = os.environ["NEO4J_USERNAME"]
if not os.environ.get("NEO4J_PASS") and os.environ.get("NEO4J_PASSWORD"):
    os.environ["NEO4J_PASS"] = os.environ["NEO4J_PASSWORD"]

NEO4J_URI = os.environ.get("NEO4J_URI")
NEO4J_USER = os.environ.get("NEO4J_USERNAME") or os.environ.get("NEO4J_USER", "neo4j")
NEO4J_DB = os.environ.get("NEO4J_DATABASE")
NEO4J_PASS = os.environ.get("NEO4J_PASSWORD") or os.environ.get("NEO4J_PASS")

if not NEO4J_URI or not NEO4J_PASS:
    raise RuntimeError(
        "NEO4J_URI / NEO4J_PASSWORD missing — set them in .env (see header)."
    )

print(f"Repo root: {REPO_ROOT}")
print(f"Neo4j URI: {NEO4J_URI}  user={NEO4J_USER}  db={NEO4J_DB or '(default)'}")
print(f"Layer 06 dir: {LAYER_DIR}")

## 2 — Neo4j: connect + show existing counts

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
driver.verify_connectivity()
print("✓ Neo4j connection OK")

_session_kwargs = {"database": NEO4J_DB} if NEO4J_DB else {}

with driver.session(**_session_kwargs) as session:
    counts = session.run(
        """
        MATCH (n)
        UNWIND labels(n) AS label
        RETURN label, count(*) AS n
        ORDER BY n DESC
        """
    )
    rows = [r.data() for r in counts]
    if rows:
        print("\nExisting node counts (will be MERGE'd, not duplicated):")
        for r in rows:
            print(f"  {r['label']:<14} {r['n']:>6}")
    else:
        print("\nGraph is empty — fresh seed coming up.")
driver.close()

## 3 — Load (or build) the layer-06 Knowledge Base

Uses the canonical builder from `notebooks/06_search_layer/yc_property_search.py`.

- **Fast path**: if a `property_knowledge_base_*.parquet` already exists under `notebooks/06_search_layer/artifacts/`, it's loaded as-is. This is what `01_build_knowledge_base.ipynb` saves on its last cell.
- **Build path**: if no parquet is present, this cell runs the full pipeline (~5 min on CPU). Requires the Layer 02 feature table + Layer 01 raw school + geocode files (see header).

In [ ]:
from yc_property_search import PropertyKnowledgeBase

artifacts_dir = LAYER_DIR / "artifacts"
existing_parquets = sorted(artifacts_dir.glob("property_knowledge_base_*.parquet"))

if existing_parquets:
    print(f"✓ Found existing KB parquet: {existing_parquets[-1].name}")
    print("  Loading instead of rebuilding (run Cell 4 in 01_build_knowledge_base.ipynb to refresh).")
    kb = PropertyKnowledgeBase()  # auto-loads latest parquet from artifacts/
else:
    print("No existing KB parquet — building from source (~5 min)...")

    # Layer 02 feature table (download via download_artifacts_from_hf.ipynb)
    feat_dir_candidates = [
        REPO_ROOT / "data" / "feature_data" / "02_feature_layer" / "training" / "outputs",
        REPO_ROOT / "hf_data" / "02_feature_layer" / "training" / "outputs",
    ]
    feature_csvs = []
    for d in feat_dir_candidates:
        if d.exists():
            feature_csvs = sorted(
                p for p in d.glob("hdb_feature_table_*.csv") if "_backup_" not in p.name
            )
            if feature_csvs:
                break
    if not feature_csvs:
        raise FileNotFoundError(
            "No hdb_feature_table_*.csv found. Run download_artifacts_from_hf.ipynb first."
        )
    feature_csv = feature_csvs[-1]

    # Layer 01 raw inputs.
    school_dir = REPO_ROOT / "notebooks" / "01_data_layer" / "raw" / "schools"
    geo_dir = REPO_ROOT / "notebooks" / "01_data_layer" / "raw" / "google_geo"
    school_info_csv = sorted(school_dir.glob("moe_general_information_of_schools_*.csv"))[-1]
    school_geocode_csv = sorted(geo_dir.glob("moe_school_geocode_*.csv"))[-1]
    hdb_geocode_csv = sorted(geo_dir.glob("hdb_geo_accessibility_noise_features_*.csv"))[-1]

    print(f"  Feature table: {feature_csv.name}")
    print(f"  School info  : {school_info_csv.name}")
    print(f"  School geo   : {school_geocode_csv.name}")
    print(f"  HDB geo      : {hdb_geocode_csv.name}")

    kb = PropertyKnowledgeBase.build(
        feature_csv=feature_csv,
        school_info_csv=school_info_csv,
        school_geocode_csv=school_geocode_csv,
        hdb_geocode_csv=hdb_geocode_csv,
        output_dir=artifacts_dir,
    )

print(f"\n{kb}")
print(f"  shape: {kb.kb.shape}")

## 4 — Push the KB into Neo4j

Reuses `kb.push_to_neo4j()` so node + edge creation matches `01_build_knowledge_base.ipynb` 1:1.

Creates:
- 26 `Town` nodes
- ~17 `FamousSchool` nodes
- ~9,710 `Property` nodes (all 12 normalized scores + raw fields as properties)
- `LOCATED_IN` (Property → Town) for every property
- `NEAREST_FAMOUS_SCHOOL` (Property → FamousSchool) for every property
- `NEAR_FAMOUS_SCHOOL` (Property → FamousSchool) for property/school pairs within 2 km

Idempotent (`MERGE` everywhere). Expected runtime: 3–8 minutes for the full graph against AuraDB.

In [ ]:
# Sanity: confirm score_school_quality survived (the parquet patch from layer 06).
sq = kb.kb.get("score_school_quality")
if sq is not None:
    print(f"score_school_quality range: {sq.min():.2f}–{sq.max():.2f}  std={sq.std():.3f}  unique={sq.nunique():,}")
    if sq.std() < 0.5:
        print("⚠️  score_school_quality looks degenerate — re-run 01_build_knowledge_base.ipynb to refresh.")

kb.push_to_neo4j(batch_size=500)

## 5 — Verify the Neo4j graph

In [ ]:
from yc_property_search import Neo4jPropertySearch

with Neo4jPropertySearch() as neo4j:
    counts = neo4j.node_counts()
    print("Node counts:")
    for label, n in counts.items():
        print(f"  {label:<14} {n:>6,}")

    rels = neo4j.graph_query(
        """
        MATCH ()-[r]->()
        RETURN type(r) AS rel, count(*) AS n
        ORDER BY n DESC
        """
    )
    print("\nRelationship counts:")
    for r in rels:
        print(f"  {r['rel']:<24} {r['n']:>6,}")

    print("\nSmoke test — top-5 education-priority Tampines flats:")
    sample = neo4j.search(
        weights={"score_famous_school": 10, "score_mrt": 5},
        filters={"town": "TAMPINES"},
        top_k=5,
    )
    cols = [
        "address_key", "flat_type", "floor_area_sqm", "resale_price",
        "dist_to_nearest_famous_school_km", "nearest_famous_school_name",
        "composite_score",
    ]
    print(sample[[c for c in cols if c in sample.columns]].to_string(index=False))

## 6 — SQLite: create tables + seed users

Reuses `backend/db.py:init_db()` to create all tables, then seeds three demo users (idempotent — existing usernames are skipped):

| Username  | Password | Display name |
|-----------|----------|--------------|
| `user`    | `1234`   | Bhuvesh      |
| `rekha`   | `1234`   | Rekha        |
| `yunchuan`| `1234`   | Yun Chuan    |

Password hashing matches production (`_hash_password` from `backend/auth_routes.py`).

In [ ]:
from backend.db import init_db, SessionLocal, _DEFAULT_DB_PATH, DATABASE_URL
from backend.auth_routes import _hash_password
from backend.sql_models import AppUser

# Users to seed (username / password / display name).
# Idempotent: existing usernames are skipped.
USERS = [
    {"username": "user",     "password": "1234", "display_name": "Bhuvesh"},
    {"username": "rekha",    "password": "1234", "display_name": "Rekha"},
    {"username": "yunchuan", "password": "1234", "display_name": "Yun Chuan"},
]

print(f"DB URL : {DATABASE_URL}")
print(f"DB file: {_DEFAULT_DB_PATH}")

init_db()
print("✓ Tables created (idempotent)\n")

with SessionLocal() as db:
    for u in USERS:
        existing = db.query(AppUser).filter(AppUser.username == u["username"]).first()
        if existing:
            print(f"  ⏭️  {u['username']:<10} already exists (display_name={existing.display_name})")
            continue
        db.add(
            AppUser(
                username=u["username"],
                password_hash=_hash_password(u["password"]),
                display_name=u["display_name"],
            )
        )
        print(f"  ✅  Created {u['username']:<10} ({u['display_name']})")
    db.commit()

    all_users = db.query(AppUser).order_by(AppUser.id).all()
    print(f"\nUsers in app_user ({len(all_users)}):")
    for u in all_users:
        print(f"  id={u.id}  username={u.username:<10}  display_name={u.display_name}")

print("\nLogin credentials (all use password 1234):")
for u in USERS:
    print(f"  {u['username']:<10}  →  {u['display_name']}")

## 7 — Done

Start the backend (`cd backend && uvicorn main:app --reload --port 8000`) and try:

```bash
# Login as demo user
curl -X POST http://localhost:8000/api/login -H 'Content-Type: application/json' \
  -d '{"username":"user","password":"1234"}'

# Hit the property-search-chat endpoint (Neo4j-backed)
curl -X POST http://localhost:8000/api/property-search-chat \
  -H 'Content-Type: application/json' -H 'Accept: text/event-stream' \
  -d '{"messages":[{"role":"user","content":"Find a 4-room flat near a famous school"}]}'
```

Re-running this notebook is safe — `MERGE` keeps Neo4j idempotent and `ensure_demo_user` is a no-op when the demo user already exists.